# AutoRA Workflow

Generated by AutoRA Workflow Editor on 2026-08-10T05:16:17.996Z

## 1. Install dependencies

In [ ]:
%pip install autora-theorist-darts==1.1.0 autora-core==5.0.3 autora-synthetic==2.2.0

## 2. Imports

In [ ]:
from autora.state import on_state, Delta, estimator_on_state, StandardState
from autora.variable import VariableCollection
from autora.experimentalist.random import pool as random_pooler, sample as random_sampler
from autora.experiment_runner.synthetic.psychophysics.stevens_power_law import stevens_power_law
from autora.theorist.darts.regressor import DARTSRegressor

import pandas as pd

## 3. Component definitions

In [ ]:
# Random Pooler
@on_state()
def random_pooler_on_state(variables: VariableCollection) -> Delta:
    return Delta(conditions=random_pooler(variables, num_samples=5, replace=True))

In [ ]:
# Random Sampler
@on_state()
def random_sampler_on_state(conditions: pd.DataFrame, num_samples: int = 1) -> Delta:
    return Delta(conditions=random_sampler(conditions=conditions, num_samples=num_samples, replace=False))

In [ ]:
# Stevens Power Law (Synthetic, Psychophysics)
runner = stevens_power_law(resolution=100, proportionality_constant=1, modality_constant=0.8, maximum_stimulus_intensity=5)

@on_state()
def stevens_power_law_on_state(conditions: pd.DataFrame) -> Delta:
    return Delta(experiment_data=runner.run(conditions=conditions, added_noise=0.01))

In [ ]:
# DARTS Regressor
darts_regressor_on_state = estimator_on_state(DARTSRegressor(batch_size=64, num_graph_nodes=2, output_type="real", classifier_weight_decay=0.01, darts_type="original", param_updates_per_epoch=10, param_updates_for_sampled_model=100, param_learning_rate_max=0.025, param_learning_rate_min=0.01, param_momentum=0.9, arch_updates_per_epoch=1, arch_learning_rate_max=0.003, arch_weight_decay=0.0001, arch_weight_decay_df=0.0003, arch_weight_decay_base=0, arch_momentum=0.9, fair_darts_loss_weight=1, max_epochs=10, grad_clip=5, primitives=["none", "add", "subtract", "linear", "linear_logistic", "linear_relu"], train_classifier_coefficients=False, train_classifier_bias=False, sampling_strategy="max"))

## 4. Run the workflow

In [ ]:
# Variables are governed by the experiment runner defined above
assert runner.variables is not None
variables = runner.variables

# Initialize state
state = StandardState(variables=variables)

# Main experiment loop (10 cycles)
for i in range(10):
    print(f'Cycle {i}')

    # Random Pooler
    state = random_pooler_on_state(state)

    # Random Sampler
    state = random_sampler_on_state(state, num_samples=1)

    # Stevens Power Law (Synthetic, Psychophysics)
    state = stevens_power_law_on_state(state)

    # DARTS Regressor
    state = darts_regressor_on_state(state)


print("Workflow completed!")
state